# 02 Sandbox

第一部分：SandboxAgent 中文快速入门。

In [2]:
import asyncio
from pathlib import Path

from agents import ModelSettings, OpenAIChatCompletionsModel, Runner
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig
from agents.sandbox.capabilities import LocalDirLazySkillSource, Shell, Skills
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient
from openai import AsyncOpenAI

client = AsyncOpenAI(
    api_key="sk-839ab4b91b25494d910a04a8812d8cf6",
    base_url="https://api.deepseek.com",
)

model = OpenAIChatCompletionsModel(
    model="deepseek-v4-pro",
    openai_client=client,
)

EXAMPLE_DIR = Path("/workspace/nas-data/huya_projects/OpenAgents/projects/ANIFORCE/notebooks/02-sandbox")
HOST_REPO_DIR = EXAMPLE_DIR / "repo"
HOST_SKILLS_DIR = EXAMPLE_DIR / "skills"


def build_agent() -> SandboxAgent[None]:
    return SandboxAgent(
        name="沙盒工程师",
        model=model,
        instructions=(
            "你是一个在沙盒工作区中修改代码的工程师。"
            "编辑文件前必须先读取 `repo/task.md`。"
            "编辑文件前必须先使用 `$credit-note-fixer` 技能。"
            "当前模型使用 Chat Completions API，不支持 apply_patch 工具；"
            "请使用 `exec_command` 通过 shell 命令查看和修改文件。"
            "只做最小正确修改，不要修改测试期望。"
            "修复后必须在 `repo/` 目录下运行 `sh tests/test_credit_note.sh`。"
            "最终回答请用中文说明修改了哪个文件、修复了什么问题、运行了什么验证命令。"
        ),
        default_manifest=Manifest(
            root=str(EXAMPLE_DIR),
            entries={
                "repo": LocalDir(),
            },
        ),
        capabilities=[
            Shell(),
            Skills(
                lazy_from=LocalDirLazySkillSource(
                    source=LocalDir(src=HOST_SKILLS_DIR),
                )
            ),
        ],
        model_settings=ModelSettings(tool_choice="auto"),
    )


result = await Runner.run(
    build_agent(),
    "请打开 `repo/task.md`，使用 `$credit-note-fixer` 技能，修复 credit note 格式化 bug，运行指定测试，并用中文总结结果。",
    max_turns=12,
    run_config=RunConfig(
        sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()),
        tracing_disabled=True,
        workflow_name="Sandbox 中文快速入门测试",
    ),
)

print("\n=== FINAL ===")
print(result.final_output)

print("\n=== NEW ITEMS ===")
for item in result.new_items:
    print(item)
    print("\n" * 2)



=== FINAL ===
测试全部通过。

---

## 修复总结

**修改文件：** `repo/credit_note.sh`

**Bug 分析：**
- 格式化标签写成了 `debit`，应为 `credit`
- 金额前硬编码了 `-`，且未处理输入为负数（如 `-12.50`）的情况，导致输出形如 `-$-12.50`

**修复内容：**
1. 使用 `${2#-}` 参数展开剥离金额参数前导的 `-`，确保金额始终为正数
2. 删除格式串中硬编码的 `-` 前缀
3. 将标签 `debit` 改为 `credit`

**验证命令：** `sh tests/test_credit_note.sh` — 输出 `2 passed`，正数和负数输入均通过。

=== NEW ITEMS ===
ReasoningItem(agent=SandboxAgent(name='沙盒工程师', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='你是一个在沙盒工作区中修改代码的工程师。编辑文件前必须先读取 `repo/task.md`。编辑文件前必须先使用 `$credit-note-fixer` 技能。当前模型使用 Chat Completions API，不支持 apply_patch 工具；请使用 `exec_command` 通过 shell 命令查看和修改文件。只做最小正确修改，不要修改测试期望。修复后必须在 `repo/` 目录下运行 `sh tests/test_credit_note.sh`。最终回答请用中文说明修改了哪个文件、修复了什么问题、运行了什么验证命令。', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f6b88c0b990>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_

In [4]:
# ##openai沙箱

# import asyncio
# from pathlib import Path

# from agents import ModelSettings, OpenAIChatCompletionsModel, Runner, OpenAIResponsesModel
# from agents.run import RunConfig
# from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig
# from agents.sandbox.capabilities import Capabilities, LocalDirLazySkillSource, Shell, Skills
# from agents.sandbox.entries import LocalDir
# from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient
# from openai import AsyncOpenAI

# client = AsyncOpenAI(
#     api_key="sk-GlkBP4qMM3FXzjBUfXximE4ypxIrtEYqVYcPXNkcwXZtYSIi",
#     base_url="https://www.codefoxai.top/v1",
# )

# model = OpenAIResponsesModel(
#     model="gpt-5.5",
#     openai_client=client,
# )

# EXAMPLE_DIR = Path("/workspace/nas-data/huya_projects/OpenAgents/projects/ANIFORCE/notebooks/02-sandbox")
# HOST_REPO_DIR = EXAMPLE_DIR / "repo"
# HOST_SKILLS_DIR = EXAMPLE_DIR / "skills"


# def build_agent() -> SandboxAgent[None]:
#     return SandboxAgent(
#         name="沙盒工程师",
#         model=model,
#         instructions=(
#             "你是一个在沙盒工作区中修改代码的工程师。"
#             "编辑文件前必须先读取 `repo/task.md`。"
#             "编辑文件前必须先使用 `$credit-note-fixer` 技能。"
#             "当前模型使用 Chat Completions API，不支持 apply_patch 工具；"
#             "请使用 `exec_command` 通过 shell 命令查看和修改文件。"
#             "只做最小正确修改，不要修改测试期望。"
#             "修复后必须在 `repo/` 目录下运行 `sh tests/test_credit_note.sh`。"
#             "最终回答请用中文说明修改了哪个文件、修复了什么问题、运行了什么验证命令。"
#         ),
#         default_manifest=Manifest(
#             root=str(EXAMPLE_DIR),
#             entries={
#                 "repo": LocalDir(),
#             },
#         ),
#         capabilities=Capabilities.default() + [
#             Skills(
#                 lazy_from=LocalDirLazySkillSource(
#                     source=LocalDir(src=HOST_SKILLS_DIR),
#                 )
#             ),
#         ],
#         model_settings=ModelSettings(tool_choice="auto"),
#     )


# result = await Runner.run(
#     build_agent(),
#     "请打开 `repo/task.md`，使用 `$credit-note-fixer` 技能，修复 credit note 格式化 bug，运行指定测试，并用中文总结结果。",
#     max_turns=12,
#     run_config=RunConfig(
#         sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()),
#         tracing_disabled=True,
#         workflow_name="Sandbox 中文快速入门测试",
#     ),
# )

# print("\n=== FINAL ===")
# print(result.final_output)

# print("\n=== NEW ITEMS ===")
# for item in result.new_items:
#     print(item)
#     print("\n" * 2)


In [4]:
import asyncio                                                                 
from pathlib import Path                                                       
                                                                                
from agents import ModelSettings, OpenAIResponsesModel, Runner                 
from agents.run import RunConfig                                               
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig            
from agents.sandbox.capabilities import LocalDirLazySkillSource, Shell, Skills 
from agents.sandbox.entries import LocalDir                                    
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient         
from openai import AsyncOpenAI                                                 
                                                                                
client = AsyncOpenAI(                                                          
    api_key="sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF",                                               
    base_url="https://api.tokenlab.sh/v1",                                     
)                                                                              
                                                                                
model = OpenAIResponsesModel(                                                  
    model="gpt-5.3-codex",                                                     
    openai_client=client,                                                      
)                                                                              
                                                                                
EXAMPLE_DIR =Path("/workspace/nas-data/huya_projects/OpenAgents/projects/ANIFORCE/notebooks/02-sandbox")                                                                      
HOST_REPO_DIR = EXAMPLE_DIR / "repo"                                           
HOST_SKILLS_DIR = EXAMPLE_DIR / "skills"                                       
                                                                                
                                                                                
def build_agent() -> SandboxAgent[None]:                                       
    return SandboxAgent(                                                       
        name="沙盒工程师",                                                     
        model=model,                                                           
        instructions=(                                                         
            "你是一个在沙盒工作区中修改代码的工程师。"                         
            "编辑文件前必须先读取 `repo/task.md`。"                            
            "编辑文件前必须先使用 `$credit-note-fixer` 技能。"                 
            "你当前使用 Responses API 模式。"                                  
            "如果有 apply_patch 工具可以使用 apply_patch；"                    
            "如果没有，请使用 `exec_command` 通过 shell 命令查看和修改文件。"  
            "只做最小正确修改，不要修改测试期望。"                             
            "修复后必须在 `repo/` 目录下运行 `sh tests/test_credit_note.sh`。" 
                                                                                
"最终回答请用中文说明修改了哪个文件、修复了什么问题、运行了什么验证命令。"       
        ),                                                                     
        default_manifest=Manifest(                                             
            root=str(EXAMPLE_DIR),                                             
            entries={                                                          
                "repo": LocalDir(),                                            
            },                                                                 
        ),                                                                     
        capabilities=[                                                         
            Shell(),                                                           
            Skills(                                                            
                lazy_from=LocalDirLazySkillSource(                             
                    source=LocalDir(src=HOST_SKILLS_DIR),                      
                )                                                              
            ),                                                                 
        ],                                                                     
        model_settings=ModelSettings(tool_choice="auto"),                      
    )                                                                          
                                                                                
                                                                                
result = await Runner.run(                                                     
    build_agent(),                                                             
    "请打开 `repo/task.md`，使用 `$credit-note-fixer` 技能，修复 credit note 格式化 bug，运行指定测试，并用中文总结结果。",                                   
    max_turns=12,                                                              
    run_config=RunConfig(                                                      
        sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()),             
        tracing_disabled=True,                                                 
        workflow_name="Sandbox Responses API 测试",                            
    ),                                                                         
)                                                                              
                                                                                
print("\n=== FINAL ===")                                                       
print(result.final_output)                                                     
                                                                                
print("\n=== NEW ITEMS ===")                                                   
for item in result.new_items:                                                  
    print(item)                                                                
    print("\n" * 2)     


=== FINAL ===
已按你的要求完成检查与验证。结果如下：

- 查看了 `repo/task.md`，并使用了 `$credit-note-fixer` 技能流程。
- 检查了 `repo/credit_note.sh:1` 和 `repo/tests/test_credit_note.sh:1`。
- 发现当前实现已经满足修复要求：  
  - 标签输出为 `credit`（不是 `debit`）  
  - 金额通过 `${2#-}` 去掉负号，始终按正数显示
- 因此**无需修改代码文件**（最小正确修改 = 0 改动）。

已运行的验证命令：

- 在 `repo/` 目录下执行：`sh tests/test_credit_note.sh`
- 测试结果：`2 passed`

=== NEW ITEMS ===
MessageOutputItem(agent=SandboxAgent(name='沙盒工程师', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='你是一个在沙盒工作区中修改代码的工程师。编辑文件前必须先读取 `repo/task.md`。编辑文件前必须先使用 `$credit-note-fixer` 技能。你当前使用 Responses API 模式。如果有 apply_patch 工具可以使用 apply_patch；如果没有，请使用 `exec_command` 通过 shell 命令查看和修改文件。只做最小正确修改，不要修改测试期望。修复后必须在 `repo/` 目录下运行 `sh tests/test_credit_note.sh`。最终回答请用中文说明修改了哪个文件、修复了什么问题、运行了什么验证命令。', prompt=None, handoffs=[], model=<agents.models.openai_responses.OpenAIResponsesModel object at 0x7fcb14a6fd90>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_p